In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

validation_df = (
    reads_df
    .withColumn(
        "declared_length",
        regexp_extract(col("header"), r"length=(\d+)", 1).cast("int")
    )
    .withColumn(
        "actual_length",
        length(col("sequence"))
    )
)

print("Podgląd danych:")
validation_df.select(
    "header",
    "declared_length",
    "actual_length"
).show(10, truncate=False)

incorrect_reads = validation_df.filter(
    col("declared_length") != col("actual_length")
)

print("Niezgodne rekordy:")
incorrect_reads.show(truncate=False)

incorrect_count = incorrect_reads.count()

print(f"Liczba rekordów z niezgodną długością: {incorrect_count}")

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

low_quality_reads = reads_df.filter(
    col("quality").contains("#")
)

print("Przykładowe odczyty o niskiej jakości:")
low_quality_reads.show(10, truncate=False)

low_quality_count = low_quality_reads.count()

print(f"Liczba odczytów zawierających '#': {low_quality_count}")

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

analysis_df = (
    reads_df
    .withColumn(
        "has_low_quality",
        col("quality").contains("#")
    )
    .withColumn(
        "seq_length",
        length(col("sequence"))
    )
    .groupBy(
        "has_low_quality",
        "seq_length"
    )
    .count()
    .orderBy(
        "has_low_quality",
        "seq_length"
    )
)

analysis_df.cache()

print("Materializacja cache...")
analysis_df.count()

print("Wyniki:")
analysis_df.show(50)

print("Liczba grup:")
analysis_df.count()

print("Liczba różnych długości:")
analysis_df.select("seq_length").distinct().count()

print("Liczba rekordów z niską jakością:")
analysis_df.filter(
    col("has_low_quality") == True
).show()